## Импорт библиотек

In [40]:
from src.config.config import DATA_FULL_PATH
from src.app.utils.db_manager import DatabaseManager
from src.app.utils.csv_analyser import CSVAnalyser

In [41]:
from warnings import filterwarnings

filterwarnings('ignore', category=UserWarning, message='.*pandas only supports SQLAlchemy connectable.*')

In [42]:
db_manager = DatabaseManager()
csv_analyser = CSVAnalyser(data_path=DATA_FULL_PATH)

В файле HomeCredit_columns_description.csv находится описание каждого столбца, эта информация нужна для определения типов данных для столбцов. Были изучены остальные csv-файлы. Для создания каждой таблицы будет следующий пайплайн:

1. загрузить CSV-файл и изучить его через pandas
2. создать таблицу и описать типы данных
3. загрузить данные из csv-файла
4. вывести данные из созданной таблицы

## Файл bureau.csv

In [43]:
table_name = "bureau"
csv_analyser.analyse_file(f"{table_name}.csv")

Название файла: bureau.csv
Данные: 1716428 строк, 17 столбцов
Столбец SK_ID_CURR
- Описание: ID of loan in our sample - one loan in our sample can have 0,1,2 or more related previous credits in credit bureau 
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 305811
- Null значений: 0 (0.00%)
- Min: 100001
- Max: 456255
Столбец SK_ID_BUREAU
- Тип данных: int64
- Уникальных значений: 1716428
- Null значений: 0 (0.00%)
- Min: 5000000
- Max: 6843457
Столбец CREDIT_ACTIVE
- Описание: Status of the Credit Bureau (CB) reported credits
- Тип данных: object
- Уникальных значений: 4
- Null значений: 0 (0.00%)
- Примеры значений: ['Closed' 'Active' 'Sold' 'Bad debt']
Столбец CREDIT_CURRENCY
- Описание: Recoded currency of the Credit Bureau credit
- Особенности: recoded
- Тип данных: object
- Уникальных значений: 4
- Null значений: 0 (0.00%)
- Примеры значений: ['currency 1' 'currency 2' 'currency 4' 'currency 3']
Столбец DAYS_CREDIT
- Описание: How many days before current applicat

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.0,NaN,0.0,0,11250.00,11250.0,0.0,0.0,Microloan,-19,NaN
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.0,-2493.0,5476.5,0,38130.84,0.0,0.0,0.0,Consumer credit,-2493,NaN
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.0,-970.0,NaN,0,15570.00,NaN,NaN,0.0,Consumer credit,-967,NaN
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.0,-1513.0,NaN,0,36000.00,0.0,0.0,0.0,Consumer credit,-1508,NaN


SQL-запрос для создания таблицы

- SK_ID_CURR, SK_ID_BUREAU - integer (Стандартный 4-байтовый тип, который может хранить значения до 6.8 млн (лимит integer - 2.1 млрд))
- CREDIT_ACTIVE - varchar(8) (самое длинное значение 'Bad debt' состоит из 8 символов)
- CREDIT_CURRENCY - varchar(11) (самое длинное значение 'currency 4' состоит из 10 символов, но лучше взять с запасом для значений 'currency 10')
- CREDIT_TYPE - varchar(50) (могут быть длинные категории)
- DAYS_CREDIT, CREDIT_DAY_OVERDUE, CNT_CREDIT_PROLONG - smallint (значения не превышают лимит типа в 32767.)
- DAYS_CREDIT_UPDATE - integer (минимальное значение -41947 выходит за нижний предел smallint)
- AMT_... и DAYS_... - real (универсальный выбор для дробных сумм и дней с NULL-значениями)

In [ ]:
table_name = "bureau"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_CURR integer,
        SK_ID_BUREAU integer,
        CREDIT_ACTIVE varchar(8),
        CREDIT_CURRENCY varchar(11),
        DAYS_CREDIT smallint,
        CREDIT_DAY_OVERDUE smallint,
        DAYS_CREDIT_ENDDATE real,
        DAYS_ENDDATE_FACT real,
        AMT_CREDIT_MAX_OVERDUE real,
        CNT_CREDIT_PROLONG smallint,
        AMT_CREDIT_SUM real,
        AMT_CREDIT_SUM_DEBT real,
        AMT_CREDIT_SUM_LIMIT real,
        AMT_CREDIT_SUM_OVERDUE real,
        CREDIT_TYPE varchar(50),
        DAYS_CREDIT_UPDATE integer,
        AMT_ANNUITY  real)
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:00


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "bureau"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_CURR,
        SK_ID_BUREAU,
        CREDIT_ACTIVE,
        CREDIT_CURRENCY,
        DAYS_CREDIT,
        CREDIT_DAY_OVERDUE,
        DAYS_CREDIT_ENDDATE,
        DAYS_ENDDATE_FACT,
        AMT_CREDIT_MAX_OVERDUE,
        CNT_CREDIT_PROLONG,
        AMT_CREDIT_SUM,
        AMT_CREDIT_SUM_DEBT,
        AMT_CREDIT_SUM_LIMIT,
        AMT_CREDIT_SUM_OVERDUE,
        CREDIT_TYPE,
        DAYS_CREDIT_UPDATE,
        AMT_ANNUITY
    )
    FROM '{DATA_FULL_PATH}/{table_name}.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:07


Вывод данных таблицы из БД

In [ ]:
table_name = "bureau"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,sk_id_bureau,credit_active,credit_currency,days_credit,credit_day_overdue,days_credit_enddate,days_enddate_fact,amt_credit_max_overdue,cnt_credit_prolong,amt_credit_sum,amt_credit_sum_debt,amt_credit_sum_limit,amt_credit_sum_overdue,credit_type,days_credit_update,amt_annuity
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.00,NaN,0.0,Consumer credit,-131,None
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.00,NaN,0.0,Credit card,-20,None
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,None
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,None
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,None
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0,180000.00,71017.38,108982.62,0.0,Credit card,-31,None
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0,42103.80,42103.80,0.00,0.0,Consumer credit,-22,None
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0,76878.45,0.00,0.00,0.0,Consumer credit,-1710,None
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0,103007.70,0.00,0.00,0.0,Consumer credit,-840,None
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0,4500.00,0.00,0.00,0.0,Credit card,-690,None


## Файл bureau_balance.csv

In [47]:
table_name = "bureau_balance"
csv_analyser.analyse_file(f"{table_name}.csv")

Название файла: bureau_balance.csv
Данные: 27299925 строк, 3 столбцов
Столбец SK_ID_BUREAU
- Тип данных: int64
- Уникальных значений: 817395
- Null значений: 0 (0.00%)
- Min: 5001709
- Max: 6842888
Столбец MONTHS_BALANCE
- Описание: Month of balance relative to application date (-1 means the freshest balance date)
- Особенности: time only relative to the application
- Тип данных: int64
- Уникальных значений: 97
- Null значений: 0 (0.00%)
- Min: -96
- Max: 0
Столбец STATUS
- Описание: Status of Credit Bureau loan during the month (active, closed, DPD0-30,… [C means closed, X means status unknown, 0 means no DPD, 1 means maximal did during month between 1-30, 2 means DPD 31-60,… 5 means DPD 120+ or sold or written off ] )
- Тип данных: object
- Уникальных значений: 8
- Null значений: 0 (0.00%)
- Примеры значений: ['C' '0' 'X' '1' '2']


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
...,...,...,...
27299920,5041336,-47,X
27299921,5041336,-48,X
27299922,5041336,-49,X
27299923,5041336,-50,X


SQL-запрос для создания таблицы

- SK_ID_BUREAU - integer (Стандартный 4-байтовый тип (лимит integer - 2.1 млрд))
- MONTHS_BALANCE - smallint (диапазон от -96 до 0)
- STATUS - char(1) (Минимальный размер для хранения одиночных символов C, X, 0-5)

In [ ]:
table_name = "bureau_balance"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_BUREAU integer,
        MONTHS_BALANCE smallint,
        STATUS char(1)
    )
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:01


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "bureau_balance"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_BUREAU,
        MONTHS_BALANCE,
        STATUS
    )
    FROM '{DATA_FULL_PATH}/{table_name}.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:22


Вывод данных таблицы из БД

In [ ]:
table_name = "bureau_balance"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_bureau,months_balance,status
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C
5,5715448,-5,C
6,5715448,-6,C
7,5715448,-7,C
8,5715448,-8,C
9,5715448,-9,0


## Файл credit_card_balance.csv

In [51]:
table_name = "credit_card_balance"
csv_analyser.analyse_file(f"{table_name}.csv")

Название файла: credit_card_balance.csv
Данные: 3840312 строк, 23 столбцов
Столбец SK_ID_PREV
- Описание: ID of previous credit in Home credit related to loan in our sample. (One loan in our sample can have 0,1,2 or more previous loans in Home Credit)
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 104307
- Null значений: 0 (0.00%)
- Min: 1000018
- Max: 2843496
Столбец SK_ID_CURR
- Описание: ID of loan in our sample
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 103558
- Null значений: 0 (0.00%)
- Min: 100006
- Max: 456250
Столбец MONTHS_BALANCE
- Описание: Month of balance relative to application date (-1 means the freshest balance date)
- Особенности: time only relative to the application
- Тип данных: int64
- Уникальных значений: 96
- Null значений: 0 (0.00%)
- Min: -96
- Max: -1
Столбец AMT_BALANCE
- Описание: Balance during the month of previous credit
- Тип данных: float64
- Уникальных значений: 1347904
- Null значений: 0 (0.00%)
- Min: -420250.

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3840307,1036507,328243,-9,0.000,45000,NaN,0.0,NaN,NaN,0.000,...,0.000,0.000,NaN,0,NaN,NaN,0.0,Active,0,0
3840308,1714892,347207,-9,0.000,45000,0.0,0.0,0.0,0.0,0.000,...,0.000,0.000,0.0,0,0.0,0.0,23.0,Active,0,0
3840309,1302323,215757,-9,275784.975,585000,270000.0,270000.0,0.0,0.0,2250.000,...,273093.975,273093.975,2.0,2,0.0,0.0,18.0,Active,0,0
3840310,1624872,430337,-10,0.000,450000,NaN,0.0,NaN,NaN,0.000,...,0.000,0.000,NaN,0,NaN,NaN,0.0,Active,0,0


SQL-запрос для создания таблицы

- SK_ID_PREV, SK_ID_CURR - integer (Стандартный 4-байтовый тип, значения до 2.8 млн входят в лимит 2.1 млрд)
- MONTHS_BALANCE - smallint (2-байтовый тип, диапазон от -96 до -1 идеально входит в лимит 32767)
- AMT_BALANCE, AMT_DRAWINGS_ATM_CURRENT, AMT_DRAWINGS_CURRENT, AMT_DRAWINGS_OTHER_CURRENT, AMT_DRAWINGS_POS_CURRENT, AMT_INST_MIN_REGULARITY, AMT_PAYMENT_CURRENT, AMT_PAYMENT_TOTAL_CURRENT, AMT_RECEIVABLE_PRINCIPAL, AMT_RECIVABLE, AMT_TOTAL_RECEIVABLE - real (4-байтовый тип для дробных сумм и корректной обработки NULL значений)
- AMT_CREDIT_LIMIT_ACTUAL - integer (Суммы до 1.35 млн без дробной части)
- CNT_DRAWINGS_ATM_CURRENT, CNT_DRAWINGS_OTHER_CURRENT, CNT_DRAWINGS_POS_CURRENT, CNT_INSTALMENT_MATURE_CUM, CNT_DRAWINGS_CURRENT - smallint (счетчики до 165 с дробной частью в файлах)
- NAME_CONTRACT_STATUS - varchar(13) (Максимальная длина строки "Sent proposal" составляет 13 символов)
- SK_DPD, SK_DPD_DEF - smallint (Количество дней просрочки до 3260)

In [ ]:
table_name = "credit_card_balance"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_PREV integer,
        SK_ID_CURR integer,
        MONTHS_BALANCE smallint,
        AMT_BALANCE real,
        AMT_CREDIT_LIMIT_ACTUAL integer,
        AMT_DRAWINGS_ATM_CURRENT real,
        AMT_DRAWINGS_CURRENT real,
        AMT_DRAWINGS_OTHER_CURRENT real,
        AMT_DRAWINGS_POS_CURRENT real,
        AMT_INST_MIN_REGULARITY real,
        AMT_PAYMENT_CURRENT real,
        AMT_PAYMENT_TOTAL_CURRENT real,
        AMT_RECEIVABLE_PRINCIPAL real,
        AMT_RECIVABLE real,
        AMT_TOTAL_RECEIVABLE real,
        CNT_DRAWINGS_ATM_CURRENT real,
        CNT_DRAWINGS_CURRENT real,
        CNT_DRAWINGS_OTHER_CURRENT real,
        CNT_DRAWINGS_POS_CURRENT real,
        CNT_INSTALMENT_MATURE_CUM real,
        NAME_CONTRACT_STATUS varchar(13),
        SK_DPD smallint,
        SK_DPD_DEF smallint
    )
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:01


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "credit_card_balance"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_PREV,
        SK_ID_CURR,
        MONTHS_BALANCE,
        AMT_BALANCE,
        AMT_CREDIT_LIMIT_ACTUAL,
        AMT_DRAWINGS_ATM_CURRENT,
        AMT_DRAWINGS_CURRENT,
        AMT_DRAWINGS_OTHER_CURRENT,
        AMT_DRAWINGS_POS_CURRENT,
        AMT_INST_MIN_REGULARITY, 
        AMT_PAYMENT_CURRENT,
        AMT_PAYMENT_TOTAL_CURRENT, 
        AMT_RECEIVABLE_PRINCIPAL,
        AMT_RECIVABLE, 
        AMT_TOTAL_RECEIVABLE,
        CNT_DRAWINGS_ATM_CURRENT, 
        CNT_DRAWINGS_CURRENT,
        CNT_DRAWINGS_OTHER_CURRENT, 
        CNT_DRAWINGS_POS_CURRENT,
        CNT_INSTALMENT_MATURE_CUM, 
        NAME_CONTRACT_STATUS,
        SK_DPD,
        SK_DPD_DEF
    )
    FROM '{DATA_FULL_PATH}/{table_name}.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:20


Вывод данных таблицы из БД

In [ ]:
table_name = "credit_card_balance"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_prev,sk_id_curr,months_balance,amt_balance,amt_credit_limit_actual,amt_drawings_atm_current,amt_drawings_current,amt_drawings_other_current,amt_drawings_pos_current,amt_inst_min_regularity,...,amt_recivable,amt_total_receivable,cnt_drawings_atm_current,cnt_drawings_current,cnt_drawings_other_current,cnt_drawings_pos_current,cnt_instalment_mature_cum,name_contract_status,sk_dpd,sk_dpd_def
0,2562384,378907,-6,56.970,135000,0.0,877.50,0.0,877.50,1700.325,...,0.000,0.000,0.0,1.0,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.00,0.0,0.00,2250.000,...,64875.555,64875.555,1.0,1.0,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.00,0.0,0.00,2250.000,...,31460.086,31460.086,0.0,0.0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.00,0.0,0.00,11795.760,...,233048.970,233048.970,1.0,1.0,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.470,450000,0.0,11547.00,0.0,11547.00,22924.890,...,453919.470,453919.470,0.0,1.0,0.0,1.0,101.0,Active,0,0
5,2646502,380010,-7,82903.810,270000,0.0,0.00,0.0,0.00,4449.105,...,82773.310,82773.310,0.0,0.0,0.0,0.0,2.0,Active,7,0
6,1079071,171320,-6,353451.660,585000,67500.0,67500.00,0.0,0.00,14684.175,...,351881.160,351881.160,1.0,1.0,0.0,0.0,6.0,Active,0,0
7,2095912,118650,-7,47962.125,45000,45000.0,45000.00,0.0,0.00,0.000,...,47962.125,47962.125,1.0,1.0,0.0,0.0,51.0,Active,0,0
8,2181852,367360,-4,291543.060,292500,90000.0,289339.44,0.0,199339.42,130.500,...,286831.560,286831.560,3.0,8.0,0.0,5.0,3.0,Active,0,0
9,1235299,203885,-5,201261.190,225000,76500.0,111026.70,0.0,34526.70,6338.340,...,197224.690,197224.690,3.0,9.0,0.0,6.0,38.0,Active,0,0


## Файл installments_payments.csv

In [55]:
table_name = "installments_payments"
csv_analyser.analyse_file(f"{table_name}.csv")

Название файла: installments_payments.csv
Данные: 13605401 строк, 8 столбцов
Столбец SK_ID_PREV
- Описание: ID of previous credit in Home credit related to loan in our sample. (One loan in our sample can have 0,1,2 or more previous loans in Home Credit)
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 997752
- Null значений: 0 (0.00%)
- Min: 1000001
- Max: 2843499
Столбец SK_ID_CURR
- Описание: ID of loan in our sample
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 339587
- Null значений: 0 (0.00%)
- Min: 100001
- Max: 456255
Столбец NUM_INSTALMENT_VERSION
- Описание: Version of installment calendar (0 is for credit card) of previous credit. Change of installment version from month to month signifies that some parameter of payment calendar has changed
- Тип данных: float64
- Уникальных значений: 65
- Null значений: 0 (0.00%)
- Min: 0.0
- Max: 178.0
Столбец NUM_INSTALMENT_NUMBER
- Описание: On which installment we observe payment
- Тип данных: int64
- У

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
...,...,...,...,...,...,...,...,...
13605396,2186857,428057,0.0,66,-1624.0,NaN,67.500,NaN
13605397,1310347,414406,0.0,47,-1539.0,NaN,67.500,NaN
13605398,1308766,402199,0.0,43,-7.0,NaN,43737.435,NaN
13605399,1062206,409297,0.0,43,-1986.0,NaN,67.500,NaN


SQL-запрос для создания таблицы

- SK_ID_PREV, SK_ID_CURR - integer (Стандартный 4-байтовый тип (лимит integer - 2.1 млрд))
- NUM_INSTALMENT_VERSION - real (В CSV значения типа 0.0, 1.0, используем real для обхода ошибок синтаксиса)
- NUM_INSTALMENT_NUMBER - smallint (2-байтовый тип, значения до 277)
- DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT - real (В CSV значения с дробной частью, также присутствуют NULL значения)
- AMT_INSTALMENT, AMT_PAYMENT - real (4-байтовый тип для дробных сумм и корректной обработки NULL значений)

In [ ]:
table_name = "installments_payments"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_PREV integer,
        SK_ID_CURR integer,
        NUM_INSTALMENT_VERSION real,
        NUM_INSTALMENT_NUMBER smallint,
        DAYS_INSTALMENT real,
        DAYS_ENTRY_PAYMENT real,
        AMT_INSTALMENT real,
        AMT_PAYMENT real
    )
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:00


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "installments_payments"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_PREV,
        SK_ID_CURR,
        NUM_INSTALMENT_VERSION,
        NUM_INSTALMENT_NUMBER,
        DAYS_INSTALMENT,
        DAYS_ENTRY_PAYMENT,
        AMT_INSTALMENT,
        AMT_PAYMENT
    )
    FROM '{DATA_FULL_PATH}/{table_name}.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:32


Вывод данных таблицы из БД

In [ ]:
table_name = "installments_payments"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_prev,sk_id_curr,num_instalment_version,num_instalment_number,days_instalment,days_entry_payment,amt_instalment,amt_payment
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
5,1137312,164489,1.0,12,-1384.0,-1417.0,5970.375,5970.375
6,2234264,184693,4.0,11,-349.0,-352.0,29432.295,29432.295
7,1818599,111420,2.0,4,-968.0,-994.0,17862.164,17862.164
8,2723183,112102,0.0,14,-197.0,-197.0,70.740,70.740
9,1413990,109741,1.0,4,-570.0,-609.0,14308.470,14308.470


## Файл POS_CASH_balance.csv

In [59]:
table_name = "POS_CASH_balance"
csv_analyser.analyse_file(f"{table_name}.csv")

Название файла: POS_CASH_balance.csv
Данные: 10001358 строк, 8 столбцов
Столбец SK_ID_PREV
- Описание: ID of previous credit in Home Credit related to loan in our sample. (One loan in our sample can have 0,1,2 or more previous loans in Home Credit)
- Тип данных: int64
- Уникальных значений: 936325
- Null значений: 0 (0.00%)
- Min: 1000001
- Max: 2843499
Столбец SK_ID_CURR
- Описание: ID of loan in our sample
- Тип данных: int64
- Уникальных значений: 337252
- Null значений: 0 (0.00%)
- Min: 100001
- Max: 456255
Столбец MONTHS_BALANCE
- Описание: Month of balance relative to application date (-1 means the information to the freshest monthly snapshot, 0 means the information at application - often it will be the same as -1 as many banks are not updating the information to Credit Bureau regularly )
- Особенности: time only relative to the application
- Тип данных: int64
- Уникальных значений: 96
- Null значений: 0 (0.00%)
- Min: -96
- Max: -1
Столбец CNT_INSTALMENT
- Описание: Term of pre

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0
...,...,...,...,...,...,...,...,...
10001353,2448283,226558,-20,6.0,0.0,Active,843,0
10001354,1717234,141565,-19,12.0,0.0,Active,602,0
10001355,1283126,315695,-21,10.0,0.0,Active,609,0
10001356,1082516,450255,-22,12.0,0.0,Active,614,0


SQL-запрос для создания таблицы

- SK_ID_PREV, SK_ID_CURR - integer (Стандартный 4-байтовый тип (лимит integer - 2.1 млрд))
- MONTHS_BALANCE - smallint (диапазон от -96 до -1)
- CNT_INSTALMENT, CNT_INSTALMENT_FUTURE - real (В CSV значения типа 1.0 (с дробной частью), а также есть NULL значения, поэтому real оптимален)
- NAME_CONTRACT_STATUS - varchar(21) (Максимальная длина строки "Returned to the store" составляет 21 символ)
- SK_DPD, SK_DPD_DEF - smallint (Дни просрочки до 4231 с запасом входят в лимит 32767)

In [ ]:
table_name = "POS_CASH_balance"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_PREV integer,
        SK_ID_CURR integer,
        MONTHS_BALANCE smallint,
        CNT_INSTALMENT real,
        CNT_INSTALMENT_FUTURE real,
        NAME_CONTRACT_STATUS varchar(21),
        SK_DPD smallint,
        SK_DPD_DEF smallint
    )
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:00


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "POS_CASH_balance"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_PREV,
        SK_ID_CURR,
        MONTHS_BALANCE,
        CNT_INSTALMENT,
        CNT_INSTALMENT_FUTURE,
        NAME_CONTRACT_STATUS,
        SK_DPD,
        SK_DPD_DEF
    )
    FROM '{DATA_FULL_PATH}/{table_name}.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:26


Вывод данных таблицы из БД

In [ ]:
table_name = "POS_CASH_balance"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_prev,sk_id_curr,months_balance,cnt_instalment,cnt_instalment_future,name_contract_status,sk_dpd,sk_dpd_def
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0
5,2207092,342166,-32,12.0,12.0,Active,0,0
6,1110516,204376,-38,48.0,43.0,Active,0,0
7,1387235,153211,-35,36.0,36.0,Active,0,0
8,1220500,112740,-31,12.0,12.0,Active,0,0
9,2371489,274851,-32,24.0,16.0,Active,0,0


## Файл previous_application.csv

In [ ]:
table_name = "previous_application"
csv_analyser.analyse_file(f"{table_name}.csv")

Название файла: previous_application.csv
Данные: 1670214 строк, 37 столбцов
Столбец SK_ID_PREV
- Описание: ID of previous credit in Home credit related to loan in our sample. (One loan in our sample can have 0,1,2 or more previous loan applications in Home Credit, previous application could, but not necessarily have to lead to credit) 
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 1670214
- Null значений: 0 (0.00%)
- Min: 1000001
- Max: 2845382
Столбец SK_ID_CURR
- Описание: ID of loan in our sample
- Особенности: hashed
- Тип данных: int64
- Уникальных значений: 338857
- Null значений: 0 (0.00%)
- Min: 100001
- Max: 456255
Столбец NAME_CONTRACT_TYPE
- Описание: Contract product type (Cash loan, consumer loan [POS] ,...) of the previous application
- Тип данных: object
- Уникальных значений: 4
- Null значений: 0 (0.00%)
- Примеры значений: ['Consumer loans' 'Cash loans' 'Revolving loans' 'XNA']
Столбец AMT_ANNUITY
- Описание: Annuity of previous application
- Тип дан

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1670209,2300464,352015,Consumer loans,14704.290,267295.5,311400.0,0.0,267295.5,WEDNESDAY,12,...,Furniture,30.0,low_normal,POS industry with interest,365243.0,-508.0,362.0,-358.0,-351.0,0.0
1670210,2357031,334635,Consumer loans,6622.020,87750.0,64291.5,29250.0,87750.0,TUESDAY,15,...,Furniture,12.0,middle,POS industry with interest,365243.0,-1604.0,-1274.0,-1304.0,-1297.0,0.0
1670211,2659632,249544,Consumer loans,11520.855,105237.0,102523.5,10525.5,105237.0,MONDAY,12,...,Consumer electronics,10.0,low_normal,POS household with interest,365243.0,-1457.0,-1187.0,-1187.0,-1181.0,0.0
1670212,2785582,400317,Cash loans,18821.520,180000.0,191880.0,NaN,180000.0,WEDNESDAY,9,...,XNA,12.0,low_normal,Cash X-Sell: low,365243.0,-1155.0,-825.0,-825.0,-817.0,1.0


SQL-запрос для создания таблицы

- SK_ID_PREV, SK_ID_CURR - integer (значения до 2.8 млн)
- NAME_CONTRACT_TYPE, NAME_CASH_LOAN_PURPOSE, NAME_CONTRACT_STATUS, NAME_PAYMENT_TYPE, CODE_REJECT_REASON, NAME_CLIENT_TYPE, NAME_GOODS_CATEGORY, NAME_PORTFOLIO, NAME_PRODUCT_TYPE, CHANNEL_TYPE, NAME_SELLER_INDUSTRY, NAME_YIELD_GROUP, PRODUCT_COMBINATION - varchar (Размер от 10 до 50 символов в зависимости от максимальной длины значения в столбце)
- AMT_ANNUITY, AMT_APPLICATION, AMT_CREDIT, AMT_DOWN_PAYMENT, AMT_GOODS_PRICE, RATE_DOWN_PAYMENT, RATE_INTEREST_PRIMARY, RATE_INTEREST_PRIVILEGED - real (4-байтовый тип для дробных значений и поддержки NULL)
- WEEKDAY_APPR_PROCESS_START - varchar(9) (Максимальная длина для "WEDNESDAY")
- HOUR_APPR_PROCESS_START, NFLAG_LAST_APPL_IN_DAY, DAYS_DECISION, SELLERPLACE_AREA - integer (Целочисленные значения, SELLERPLACE_AREA до 4 млн)
- FLAG_LAST_APPL_PER_CONTRACT - char(1) (Для флагов Y/N)
- CNT_PAYMENT, DAYS_FIRST_DRAWING, DAYS_FIRST_DUE, DAYS_LAST_DUE_1ST_VERSION, DAYS_LAST_DUE, DAYS_TERMINATION, NFLAG_INSURED_ON_APPROVAL - real (В CSV значения с точкой типа 365243.0 и много NULL)
- NAME_TYPE_SUITE - varchar(15) (Максимальная длина для "Unaccompanied")

In [ ]:
table_name = "previous_application"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_PREV integer,
        SK_ID_CURR integer,
        NAME_CONTRACT_TYPE varchar(15),
        AMT_ANNUITY real,
        AMT_APPLICATION real,
        AMT_CREDIT real,
        AMT_DOWN_PAYMENT real,
        AMT_GOODS_PRICE real,
        WEEKDAY_APPR_PROCESS_START varchar(9),
        HOUR_APPR_PROCESS_START integer,
        FLAG_LAST_APPL_PER_CONTRACT char(1),
        NFLAG_LAST_APPL_IN_DAY integer,
        RATE_DOWN_PAYMENT real,
        RATE_INTEREST_PRIMARY real,
        RATE_INTEREST_PRIVILEGED real,
        NAME_CASH_LOAN_PURPOSE varchar(34),
        NAME_CONTRACT_STATUS varchar(15),
        DAYS_DECISION integer,
        NAME_PAYMENT_TYPE varchar(50),
        CODE_REJECT_REASON varchar(9),
        NAME_TYPE_SUITE varchar(15),
        NAME_CLIENT_TYPE varchar(10),
        NAME_GOODS_CATEGORY varchar(25),
        NAME_PORTFOLIO varchar(10),
        NAME_PRODUCT_TYPE varchar(10),
        CHANNEL_TYPE varchar(30),
        SELLERPLACE_AREA integer,
        NAME_SELLER_INDUSTRY varchar(25),
        CNT_PAYMENT real,
        NAME_YIELD_GROUP varchar(15),
        PRODUCT_COMBINATION varchar(34),
        DAYS_FIRST_DRAWING real,
        DAYS_FIRST_DUE real,
        DAYS_LAST_DUE_1ST_VERSION real,
        DAYS_LAST_DUE real,
        DAYS_TERMINATION real,
        NFLAG_INSURED_ON_APPROVAL real
    )
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:00


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "previous_application"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_PREV,
        SK_ID_CURR,
        NAME_CONTRACT_TYPE,
        AMT_ANNUITY,
        AMT_APPLICATION,
        AMT_CREDIT,
        AMT_DOWN_PAYMENT,
        AMT_GOODS_PRICE,
        WEEKDAY_APPR_PROCESS_START,
        HOUR_APPR_PROCESS_START,
        FLAG_LAST_APPL_PER_CONTRACT,
        NFLAG_LAST_APPL_IN_DAY,
        RATE_DOWN_PAYMENT,
        RATE_INTEREST_PRIMARY,
        RATE_INTEREST_PRIVILEGED,
        NAME_CASH_LOAN_PURPOSE,
        NAME_CONTRACT_STATUS,
        DAYS_DECISION,
        NAME_PAYMENT_TYPE,
        CODE_REJECT_REASON,
        NAME_TYPE_SUITE,
        NAME_CLIENT_TYPE,
        NAME_GOODS_CATEGORY,
        NAME_PORTFOLIO,
        NAME_PRODUCT_TYPE,
        CHANNEL_TYPE,
        SELLERPLACE_AREA,
        NAME_SELLER_INDUSTRY,
        CNT_PAYMENT,
        NAME_YIELD_GROUP,
        PRODUCT_COMBINATION,
        DAYS_FIRST_DRAWING,
        DAYS_FIRST_DUE,
        DAYS_LAST_DUE_1ST_VERSION,
        DAYS_LAST_DUE,
        DAYS_TERMINATION,
        NFLAG_INSURED_ON_APPROVAL
    )
    FROM '{DATA_FULL_PATH}/{table_name}.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:13


Вывод данных таблицы из БД

In [ ]:
table_name = "previous_application"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_prev,sk_id_curr,name_contract_type,amt_annuity,amt_application,amt_credit,amt_down_payment,amt_goods_price,weekday_appr_process_start,hour_appr_process_start,...,name_seller_industry,cnt_payment,name_yield_group,product_combination,days_first_drawing,days_first_due,days_last_due_1st_version,days_last_due,days_termination,nflag_insured_on_approval
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.336,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN
5,1383531,199383,Cash loans,23703.930,315000.0,340573.5,NaN,315000.0,SATURDAY,8,...,XNA,18.0,low_normal,Cash X-Sell: low,365243.0,-654.0,-144.0,-144.0,-137.0,1.0
6,2315218,175704,Cash loans,NaN,0.0,0.0,NaN,NaN,TUESDAY,11,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
7,1656711,296299,Cash loans,NaN,0.0,0.0,NaN,NaN,MONDAY,7,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
8,2367563,342292,Cash loans,NaN,0.0,0.0,NaN,NaN,MONDAY,15,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN
9,2579447,334349,Cash loans,NaN,0.0,0.0,NaN,NaN,SATURDAY,15,...,XNA,NaN,XNA,Cash,NaN,NaN,NaN,NaN,NaN,NaN


## Файлы application_test.csv и application_train.csv

In [67]:
table_name = "application"
csv_analyser.analyse_file(f"{table_name}_train.csv")

Название файла: application_train.csv
Данные: 307511 строк, 122 столбцов
Столбец SK_ID_CURR
- Тип данных: int64
- Уникальных значений: 307511
- Null значений: 0 (0.00%)
- Min: 100002
- Max: 456255
Столбец TARGET
- Тип данных: int64
- Уникальных значений: 2
- Null значений: 0 (0.00%)
- Min: 0
- Max: 1
Столбец NAME_CONTRACT_TYPE
- Тип данных: object
- Уникальных значений: 2
- Null значений: 0 (0.00%)
- Примеры значений: ['Cash loans' 'Revolving loans']
Столбец CODE_GENDER
- Тип данных: object
- Уникальных значений: 3
- Null значений: 0 (0.00%)
- Примеры значений: ['M' 'F' 'XNA']
Столбец FLAG_OWN_CAR
- Тип данных: object
- Уникальных значений: 2
- Null значений: 0 (0.00%)
- Примеры значений: ['N' 'Y']
Столбец FLAG_OWN_REALTY
- Тип данных: object
- Уникальных значений: 2
- Null значений: 0 (0.00%)
- Примеры значений: ['Y' 'N']
Столбец CNT_CHILDREN
- Тип данных: int64
- Уникальных значений: 15
- Null значений: 0 (0.00%)
- Min: 0
- Max: 19
Столбец AMT_INCOME_TOTAL
- Тип данных: float64
- Уни

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307506,456251,0,Cash loans,M,N,N,0,157500.0,254700.0,27558.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307507,456252,0,Cash loans,F,N,Y,0,72000.0,269550.0,12001.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
307508,456253,0,Cash loans,F,N,Y,0,153000.0,677664.0,29979.0,...,0,0,0,0,1.0,0.0,0.0,1.0,0.0,1.0
307509,456254,1,Cash loans,F,N,Y,0,171000.0,370107.0,20205.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


SQL-запрос для создания таблицы

- SK_ID_CURR - integer (лимит 2.1 млрд)
- NAME_CONTRACT_TYPE, NAME_TYPE_SUITE, NAME_INCOME_TYPE, NAME_EDUCATION_TYPE, NAME_FAMILY_STATUS, NAME_HOUSING_TYPE, OCCUPATION_TYPE, WEEKDAY_APPR_PROCESS_START, ORGANIZATION_TYPE, FONDKAPREMONT_MODE, HOUSETYPE_MODE, WALLSMATERIAL_MODE - varchar (Размер от 10 до 60 символов, ORGANIZATION_TYPE - самый длинный до 60)
- CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, FLAG_LAST_APPL_PER_CONTRACT, EMERGENCYSTATE_MODE - varchar(5) (Для коротких текстовых меток и флагов)
- AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT_GOODS_PRICE, REGION_POPULATION_RELATIVE, OWN_CAR_AGE, EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3, ..._AVG, ..._MODE, ..._MEDI, TOTALAREA_MODE - real (для всех дробных показателей, коэффициентов и сумм)
- DAYS_BIRTH, DAYS_EMPLOYED, DAYS_ID_PUBLISH, HOUR_APPR_PROCESS_START - integer (Целые числа)
- CNT_CHILDREN, CNT_FAM_MEMBERS, REGION_RATING_CLIENT, REGION_RATING_CLIENT_W_CITY, ...CNT_SOCIAL_CIRCLE, AMT_REQ_CREDIT_BUREAU... - real (Используем real, так как в CSV эти счетчики часто идут в формате "0.0" и содержат NULL)
- DAYS_REGISTRATION, DAYS_LAST_PHONE_CHANGE - real (В CSV значения с дробной частью)
- TARGET, FLAG_MOBIL, FLAG_EMP_PHONE, FLAG_WORK_PHONE, FLAG_CONT_MOBILE, FLAG_PHONE, FLAG_EMAIL, REG_REGION_..., LIVE_REGION_..., REG_CITY_..., LIVE_CITY_..., FLAG_DOCUMENT_... - smallint (Бинарные флаги 0/1)

In [ ]:
table_name = "application"
sql_schema_query = f"""
    DROP TABLE IF EXISTS {table_name}; 
    CREATE TABLE {table_name}(
        SK_ID_CURR integer,
        TARGET smallint,
        NAME_CONTRACT_TYPE varchar(20),
        CODE_GENDER varchar(5),
        FLAG_OWN_CAR varchar(5),
        FLAG_OWN_REALTY varchar(5),
        CNT_CHILDREN real,
        AMT_INCOME_TOTAL real,
        AMT_CREDIT real,
        AMT_ANNUITY real,
        AMT_GOODS_PRICE real,
        NAME_TYPE_SUITE varchar(25),
        NAME_INCOME_TYPE varchar(25),
        NAME_EDUCATION_TYPE varchar(40),
        NAME_FAMILY_STATUS varchar(25),
        NAME_HOUSING_TYPE varchar(25),
        REGION_POPULATION_RELATIVE real,
        DAYS_BIRTH integer,
        DAYS_EMPLOYED integer,
        DAYS_REGISTRATION real,
        DAYS_ID_PUBLISH integer,
        OWN_CAR_AGE real,
        FLAG_MOBIL smallint,
        FLAG_EMP_PHONE smallint,
        FLAG_WORK_PHONE smallint,
        FLAG_CONT_MOBILE smallint,
        FLAG_PHONE smallint,
        FLAG_EMAIL smallint,
        OCCUPATION_TYPE varchar(30),
        CNT_FAM_MEMBERS real,
        REGION_RATING_CLIENT real,
        REGION_RATING_CLIENT_W_CITY real,
        WEEKDAY_APPR_PROCESS_START varchar(15),
        HOUR_APPR_PROCESS_START integer,
        REG_REGION_NOT_LIVE_REGION smallint,
        REG_REGION_NOT_WORK_REGION smallint,
        LIVE_REGION_NOT_WORK_REGION smallint,
        REG_CITY_NOT_LIVE_CITY smallint,
        REG_CITY_NOT_WORK_CITY smallint,
        LIVE_CITY_NOT_WORK_CITY smallint,
        ORGANIZATION_TYPE varchar(60),
        EXT_SOURCE_1 real,
        EXT_SOURCE_2 real,
        EXT_SOURCE_3 real,
        APARTMENTS_AVG real,
        BASEMENTAREA_AVG real,
        YEARS_BEGINEXPLUATATION_AVG real,
        YEARS_BUILD_AVG real,
        COMMONAREA_AVG real,
        ELEVATORS_AVG real,
        ENTRANCES_AVG real,
        FLOORSMAX_AVG real,
        FLOORSMIN_AVG real,
        LANDAREA_AVG real,
        LIVINGAPARTMENTS_AVG real,
        LIVINGAREA_AVG real,
        NONLIVINGAPARTMENTS_AVG real,
        NONLIVINGAREA_AVG real,
        APARTMENTS_MODE real,
        BASEMENTAREA_MODE real,
        YEARS_BEGINEXPLUATATION_MODE real,
        YEARS_BUILD_MODE real,
        COMMONAREA_MODE real,
        ELEVATORS_MODE real,
        ENTRANCES_MODE real,
        FLOORSMAX_MODE real,
        FLOORSMIN_MODE real,
        LANDAREA_MODE real,
        LIVINGAPARTMENTS_MODE real,
        LIVINGAREA_MODE real,
        NONLIVINGAPARTMENTS_MODE real,
        NONLIVINGAREA_MODE real,
        APARTMENTS_MEDI real,
        BASEMENTAREA_MEDI real,
        YEARS_BEGINEXPLUATATION_MEDI real,
        YEARS_BUILD_MEDI real,
        COMMONAREA_MEDI real,
        ELEVATORS_MEDI real,
        ENTRANCES_MEDI real,
        FLOORSMAX_MEDI real,
        FLOORSMIN_MEDI real,
        LANDAREA_MEDI real,
        LIVINGAPARTMENTS_MEDI real,
        LIVINGAREA_MEDI real,
        NONLIVINGAPARTMENTS_MEDI real,
        NONLIVINGAREA_MEDI real,
        FONDKAPREMONT_MODE varchar(30),
        HOUSETYPE_MODE varchar(30),
        TOTALAREA_MODE real,
        WALLSMATERIAL_MODE varchar(25),
        EMERGENCYSTATE_MODE varchar(10),
        OBS_30_CNT_SOCIAL_CIRCLE real,
        DEF_30_CNT_SOCIAL_CIRCLE real,
        OBS_60_CNT_SOCIAL_CIRCLE real,
        DEF_60_CNT_SOCIAL_CIRCLE real,
        DAYS_LAST_PHONE_CHANGE real,
        FLAG_DOCUMENT_2 smallint,
        FLAG_DOCUMENT_3 smallint,
        FLAG_DOCUMENT_4 smallint,
        FLAG_DOCUMENT_5 smallint,
        FLAG_DOCUMENT_6 smallint,
        FLAG_DOCUMENT_7 smallint,
        FLAG_DOCUMENT_8 smallint,
        FLAG_DOCUMENT_9 smallint,
        FLAG_DOCUMENT_10 smallint,
        FLAG_DOCUMENT_11 smallint,
        FLAG_DOCUMENT_12 smallint,
        FLAG_DOCUMENT_13 smallint,
        FLAG_DOCUMENT_14 smallint,
        FLAG_DOCUMENT_15 smallint,
        FLAG_DOCUMENT_16 smallint,
        FLAG_DOCUMENT_17 smallint,
        FLAG_DOCUMENT_18 smallint,
        FLAG_DOCUMENT_19 smallint,
        FLAG_DOCUMENT_20 smallint,
        FLAG_DOCUMENT_21 smallint,
        AMT_REQ_CREDIT_BUREAU_HOUR real,
        AMT_REQ_CREDIT_BUREAU_DAY real,
        AMT_REQ_CREDIT_BUREAU_WEEK real,
        AMT_REQ_CREDIT_BUREAU_MON real,
        AMT_REQ_CREDIT_BUREAU_QRT real,
        AMT_REQ_CREDIT_BUREAU_YEAR real
    )
"""

db_manager.send_sql_query(sql_schema_query)

Время выполнения: 0:00:00


SQL-запрос для заполнения таблицы данными из csv-файла

In [ ]:
table_name = "application"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_CURR,
        TARGET,
        NAME_CONTRACT_TYPE,
        CODE_GENDER,
        FLAG_OWN_CAR,
        FLAG_OWN_REALTY,
        CNT_CHILDREN,
        AMT_INCOME_TOTAL,
        AMT_CREDIT,
        AMT_ANNUITY,
        AMT_GOODS_PRICE,
        NAME_TYPE_SUITE,
        NAME_INCOME_TYPE,
        NAME_EDUCATION_TYPE,
        NAME_FAMILY_STATUS,
        NAME_HOUSING_TYPE,
        REGION_POPULATION_RELATIVE,
        DAYS_BIRTH,
        DAYS_EMPLOYED,
        DAYS_REGISTRATION,
        DAYS_ID_PUBLISH,
        OWN_CAR_AGE,
        FLAG_MOBIL,
        FLAG_EMP_PHONE,
        FLAG_WORK_PHONE,
        FLAG_CONT_MOBILE,
        FLAG_PHONE,
        FLAG_EMAIL,
        OCCUPATION_TYPE,
        CNT_FAM_MEMBERS,
        REGION_RATING_CLIENT,
        REGION_RATING_CLIENT_W_CITY,
        WEEKDAY_APPR_PROCESS_START,
        HOUR_APPR_PROCESS_START,
        REG_REGION_NOT_LIVE_REGION,
        REG_REGION_NOT_WORK_REGION,
        LIVE_REGION_NOT_WORK_REGION,
        REG_CITY_NOT_LIVE_CITY,
        REG_CITY_NOT_WORK_CITY,
        LIVE_CITY_NOT_WORK_CITY,
        ORGANIZATION_TYPE,
        EXT_SOURCE_1,
        EXT_SOURCE_2,
        EXT_SOURCE_3,
        APARTMENTS_AVG,
        BASEMENTAREA_AVG,
        YEARS_BEGINEXPLUATATION_AVG,
        YEARS_BUILD_AVG,
        COMMONAREA_AVG,
        ELEVATORS_AVG,
        ENTRANCES_AVG,
        FLOORSMAX_AVG,
        FLOORSMIN_AVG,
        LANDAREA_AVG,
        LIVINGAPARTMENTS_AVG,
        LIVINGAREA_AVG,
        NONLIVINGAPARTMENTS_AVG,
        NONLIVINGAREA_AVG,
        APARTMENTS_MODE,
        BASEMENTAREA_MODE,
        YEARS_BEGINEXPLUATATION_MODE,
        YEARS_BUILD_MODE,
        COMMONAREA_MODE,
        ELEVATORS_MODE,
        ENTRANCES_MODE,
        FLOORSMAX_MODE,
        FLOORSMIN_MODE,
        LANDAREA_MODE,
        LIVINGAPARTMENTS_MODE,
        LIVINGAREA_MODE,
        NONLIVINGAPARTMENTS_MODE,
        NONLIVINGAREA_MODE,
        APARTMENTS_MEDI,
        BASEMENTAREA_MEDI,
        YEARS_BEGINEXPLUATATION_MEDI,
        YEARS_BUILD_MEDI,
        COMMONAREA_MEDI,
        ELEVATORS_MEDI,
        ENTRANCES_MEDI,
        FLOORSMAX_MEDI,
        FLOORSMIN_MEDI,
        LANDAREA_MEDI,
        LIVINGAPARTMENTS_MEDI,
        LIVINGAREA_MEDI,
        NONLIVINGAPARTMENTS_MEDI,
        NONLIVINGAREA_MEDI,
        FONDKAPREMONT_MODE,
        HOUSETYPE_MODE,
        TOTALAREA_MODE,
        WALLSMATERIAL_MODE,
        EMERGENCYSTATE_MODE,
        OBS_30_CNT_SOCIAL_CIRCLE,
        DEF_30_CNT_SOCIAL_CIRCLE,
        OBS_60_CNT_SOCIAL_CIRCLE,
        DEF_60_CNT_SOCIAL_CIRCLE,
        DAYS_LAST_PHONE_CHANGE,
        FLAG_DOCUMENT_2,
        FLAG_DOCUMENT_3,
        FLAG_DOCUMENT_4,
        FLAG_DOCUMENT_5,
        FLAG_DOCUMENT_6,
        FLAG_DOCUMENT_7,
        FLAG_DOCUMENT_8,
        FLAG_DOCUMENT_9,
        FLAG_DOCUMENT_10,
        FLAG_DOCUMENT_11,
        FLAG_DOCUMENT_12,
        FLAG_DOCUMENT_13,
        FLAG_DOCUMENT_14,
        FLAG_DOCUMENT_15,
        FLAG_DOCUMENT_16,
        FLAG_DOCUMENT_17,
        FLAG_DOCUMENT_18,
        FLAG_DOCUMENT_19,
        FLAG_DOCUMENT_20,
        FLAG_DOCUMENT_21,
        AMT_REQ_CREDIT_BUREAU_HOUR,
        AMT_REQ_CREDIT_BUREAU_DAY,
        AMT_REQ_CREDIT_BUREAU_WEEK,
        AMT_REQ_CREDIT_BUREAU_MON,
        AMT_REQ_CREDIT_BUREAU_QRT,
        AMT_REQ_CREDIT_BUREAU_YEAR
    )
    FROM '{DATA_FULL_PATH}/{table_name}_train.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:06


In [ ]:
table_name = "application"
sql_data_query = f"""
    COPY {table_name}(
        SK_ID_CURR,
        NAME_CONTRACT_TYPE,
        CODE_GENDER,
        FLAG_OWN_CAR,
        FLAG_OWN_REALTY,
        CNT_CHILDREN,
        AMT_INCOME_TOTAL,
        AMT_CREDIT,
        AMT_ANNUITY,
        AMT_GOODS_PRICE,
        NAME_TYPE_SUITE,
        NAME_INCOME_TYPE,
        NAME_EDUCATION_TYPE,
        NAME_FAMILY_STATUS,
        NAME_HOUSING_TYPE,
        REGION_POPULATION_RELATIVE,
        DAYS_BIRTH,
        DAYS_EMPLOYED,
        DAYS_REGISTRATION,
        DAYS_ID_PUBLISH,
        OWN_CAR_AGE,
        FLAG_MOBIL,
        FLAG_EMP_PHONE,
        FLAG_WORK_PHONE,
        FLAG_CONT_MOBILE,
        FLAG_PHONE,
        FLAG_EMAIL,
        OCCUPATION_TYPE,
        CNT_FAM_MEMBERS,
        REGION_RATING_CLIENT,
        REGION_RATING_CLIENT_W_CITY,
        WEEKDAY_APPR_PROCESS_START,
        HOUR_APPR_PROCESS_START,
        REG_REGION_NOT_LIVE_REGION,
        REG_REGION_NOT_WORK_REGION,
        LIVE_REGION_NOT_WORK_REGION,
        REG_CITY_NOT_LIVE_CITY,
        REG_CITY_NOT_WORK_CITY,
        LIVE_CITY_NOT_WORK_CITY,
        ORGANIZATION_TYPE,
        EXT_SOURCE_1,
        EXT_SOURCE_2,
        EXT_SOURCE_3,
        APARTMENTS_AVG,
        BASEMENTAREA_AVG,
        YEARS_BEGINEXPLUATATION_AVG,
        YEARS_BUILD_AVG,
        COMMONAREA_AVG,
        ELEVATORS_AVG,
        ENTRANCES_AVG,
        FLOORSMAX_AVG,
        FLOORSMIN_AVG,
        LANDAREA_AVG,
        LIVINGAPARTMENTS_AVG,
        LIVINGAREA_AVG,
        NONLIVINGAPARTMENTS_AVG,
        NONLIVINGAREA_AVG,
        APARTMENTS_MODE,
        BASEMENTAREA_MODE,
        YEARS_BEGINEXPLUATATION_MODE,
        YEARS_BUILD_MODE,
        COMMONAREA_MODE,
        ELEVATORS_MODE,
        ENTRANCES_MODE,
        FLOORSMAX_MODE,
        FLOORSMIN_MODE,
        LANDAREA_MODE,
        LIVINGAPARTMENTS_MODE,
        LIVINGAREA_MODE,
        NONLIVINGAPARTMENTS_MODE,
        NONLIVINGAREA_MODE,
        APARTMENTS_MEDI,
        BASEMENTAREA_MEDI,
        YEARS_BEGINEXPLUATATION_MEDI,
        YEARS_BUILD_MEDI,
        COMMONAREA_MEDI,
        ELEVATORS_MEDI,
        ENTRANCES_MEDI,
        FLOORSMAX_MEDI,
        FLOORSMIN_MEDI,
        LANDAREA_MEDI,
        LIVINGAPARTMENTS_MEDI,
        LIVINGAREA_MEDI,
        NONLIVINGAPARTMENTS_MEDI,
        NONLIVINGAREA_MEDI,
        FONDKAPREMONT_MODE,
        HOUSETYPE_MODE,
        TOTALAREA_MODE,
        WALLSMATERIAL_MODE,
        EMERGENCYSTATE_MODE,
        OBS_30_CNT_SOCIAL_CIRCLE,
        DEF_30_CNT_SOCIAL_CIRCLE,
        OBS_60_CNT_SOCIAL_CIRCLE,
        DEF_60_CNT_SOCIAL_CIRCLE,
        DAYS_LAST_PHONE_CHANGE,
        FLAG_DOCUMENT_2,
        FLAG_DOCUMENT_3,
        FLAG_DOCUMENT_4,
        FLAG_DOCUMENT_5,
        FLAG_DOCUMENT_6,
        FLAG_DOCUMENT_7,
        FLAG_DOCUMENT_8,
        FLAG_DOCUMENT_9,
        FLAG_DOCUMENT_10,
        FLAG_DOCUMENT_11,
        FLAG_DOCUMENT_12,
        FLAG_DOCUMENT_13,
        FLAG_DOCUMENT_14,
        FLAG_DOCUMENT_15,
        FLAG_DOCUMENT_16,
        FLAG_DOCUMENT_17,
        FLAG_DOCUMENT_18,
        FLAG_DOCUMENT_19,
        FLAG_DOCUMENT_20,
        FLAG_DOCUMENT_21,
        AMT_REQ_CREDIT_BUREAU_HOUR,
        AMT_REQ_CREDIT_BUREAU_DAY,
        AMT_REQ_CREDIT_BUREAU_WEEK,
        AMT_REQ_CREDIT_BUREAU_MON,
        AMT_REQ_CREDIT_BUREAU_QRT,
        AMT_REQ_CREDIT_BUREAU_YEAR
    )
    FROM '{DATA_FULL_PATH}/{table_name}_test.csv' DELIMITER ',' CSV HEADER;
"""

db_manager.send_sql_query(sql_data_query)

Время выполнения: 0:00:01


Вывод данных таблицы из БД

In [ ]:
table_name = "application"
sql_query = f"""
    SELECT * FROM {table_name}
    LIMIT 10
"""

df = db_manager.get_df_from_query(sql_query)
df

Время выполнения: 0:00:00


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,100002,1,Cash loans,M,N,Y,0.0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0.0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0.0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0.0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0.0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
5,100008,0,Cash loans,M,N,Y,0.0,99000.0,490495.5,27517.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
6,100009,0,Cash loans,F,Y,Y,1.0,171000.0,1560726.0,41301.0,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0
7,100010,0,Cash loans,M,Y,Y,0.0,360000.0,1530000.0,42075.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
8,100011,0,Cash loans,F,N,Y,0.0,112500.0,1019610.0,33826.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
9,100012,0,Revolving loans,M,N,Y,0.0,135000.0,405000.0,20250.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## Создадим некоторые индексы для оптимизации запросов, а также определим первичные и внешние ключи

In [72]:
sql_keys_query = """
    ALTER TABLE application ADD PRIMARY KEY (SK_ID_CURR);

    ALTER TABLE bureau ADD PRIMARY KEY (SK_ID_BUREAU);

    ALTER TABLE bureau_balance ADD PRIMARY KEY (SK_ID_BUREAU, MONTHS_BALANCE);

    ALTER TABLE pos_cash_balance ADD PRIMARY KEY (SK_ID_PREV, MONTHS_BALANCE);
    
    ALTER TABLE credit_card_balance ADD PRIMARY KEY (SK_ID_PREV, MONTHS_BALANCE);

    ALTER TABLE previous_application ADD PRIMARY KEY (SK_ID_PREV);
"""

db_manager.send_sql_query(sql_keys_query)

Время выполнения: 0:01:56


Для таблиц вместо внешних ключей лучше создать индексы, чтобы не было ошибок при импорте информации оттуда из-за отсутствия значений внешних ключей.

- Некоторые индексы были созданы вместо внешних ключей, потому что при создании внешнего ключа возникала ошибка из-за пропусков в данных (idx_bureau_bal_sk_id_bureau и др.)
- Индексы idx_app_target и idx_app_income_type ускоряют запросы WHERE и группировки по этим полям
- Индексы на SK_ID_CURR в дочерних таблицах нужны для оптимизации операций JOIN с таблицей application
- Часто индексы используются для статусов, потому что они могут использоваться в WHERE во многих запросах

In [73]:
sql_index_query = """
    CREATE INDEX IF NOT EXISTS idx_bureau_bal_sk_id_bureau ON bureau_balance(SK_ID_BUREAU);

    CREATE INDEX IF NOT EXISTS idx_app ON application(TARGET);
    CREATE INDEX IF NOT EXISTS idx_app_income_type ON application(NAME_INCOME_TYPE);

    CREATE INDEX IF NOT EXISTS idx_bureau_curr ON bureau(SK_ID_CURR);
    CREATE INDEX IF NOT EXISTS idx_prev_app_curr ON previous_application(SK_ID_CURR);
    CREATE INDEX IF NOT EXISTS idx_pos_cash_curr ON pos_cash_balance(SK_ID_CURR);
    CREATE INDEX IF NOT EXISTS idx_inst_pay_curr ON installments_payments(SK_ID_CURR);
    CREATE INDEX IF NOT EXISTS idx_cc_bal_curr ON credit_card_balance(SK_ID_CURR);

    CREATE INDEX IF NOT EXISTS idx_prev_app_status ON previous_application(NAME_CONTRACT_STATUS);
    CREATE INDEX IF NOT EXISTS idx_bureau_active ON bureau(CREDIT_ACTIVE);
"""

db_manager.send_sql_query(sql_index_query)

Время выполнения: 0:01:12
